In [1]:
#--import libraries
import numpy as np
import math

In [2]:
#--define q,k,v
seq_length=4
d_q=8
d_k=8
d_v=8

q=np.random.randn(seq_length, d_q)
k=np.random.randn(seq_length, d_k)
v=np.random.randn(seq_length, d_v)

print(f'q vector :{q} \n with var :{q.var()} \n and mean :{q.mean()}')
print(f'k vector :{k} \n with var :{k.var()} \n and mean :{k.mean()}')
print(f'v vector :{v} \n with var :{v.var()} \n and mean :{v.mean()}')

q vector :[[-1.05868046e+00  1.13353622e+00 -1.15197075e+00  2.30283757e+00
   6.99061056e-01  4.02627229e-03 -4.49626017e-01 -1.31281438e+00]
 [ 1.36866800e-01  1.16701087e+00 -1.57971755e-01  6.48436901e-02
   1.27611137e+00  5.91425506e-01 -8.03834562e-01 -4.33774414e-01]
 [ 1.00063230e-01  2.13128072e-01  2.14050104e-03  1.46966142e+00
   1.03523569e+00  6.03602151e-01  2.07821611e-01  6.68158515e-01]
 [ 2.04120692e+00  1.79199483e+00 -2.14671529e+00 -3.26385953e-01
   1.37733454e-01  4.44679254e-01  7.69278572e-01 -1.17185587e+00]] 
 with var :1.005119861952768 
 and mean :0.2452123160970215
k vector :[[ 1.42186993  0.01819524  0.6633115  -0.86308867  0.89745129  0.93695734
   0.68728597  1.21006302]
 [-0.06502169  0.90126928  0.67751007  1.02175077 -0.19063354  0.16205701
  -0.80725845  2.3986433 ]
 [ 0.92054433  1.08100705  0.41546803 -1.93894139 -1.04223758 -0.15827318
   0.65196064  0.9899678 ]
 [ 0.64674527  0.3076241  -0.91039099  1.0818314   1.18150477 -0.98498394
  -0.0073

In [3]:
#---dot product
# k,k.T
dot_product=np.matmul(q,k.T)
print(f'dot_product :{dot_product} with shape :{dot_product.shape}') #---> (seq_len,seq_len)

dot_product :[[-5.5028135  -0.25570817 -7.01488312  4.06874061]
 [ 0.67712017  0.46312555 -1.18093061  1.60558483]
 [ 1.32509638  3.02403292 -2.90374111  2.32525959]
 [ 1.44364507 -3.69162324  2.68464018  3.22696527]] with shape :(4, 4)


In [4]:
#--why scaling
not_scaled=dot_product
scaled=dot_product/math.sqrt(d_k)

print(f'without scaling dot_product var :{not_scaled.var()}')
print(f'with scaling dot_product var :{scaled.var()}')

without scaling dot_product var :9.926351575145791
with scaling dot_product var :1.2407939468932236


In [5]:
#--create mask
mask=np.ones((seq_length,seq_length))
print(f'mask :{mask}')
mask=np.tril(mask)
print(f'mask :{mask}')
mask[mask==0]=float('-inf')
print(f'mask :{mask}')
mask[mask==1]=0
print(f'mask :{mask}')


mask :[[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]]
mask :[[1. 0. 0. 0.]
 [1. 1. 0. 0.]
 [1. 1. 1. 0.]
 [1. 1. 1. 1.]]
mask :[[  1. -inf -inf -inf]
 [  1.   1. -inf -inf]
 [  1.   1.   1. -inf]
 [  1.   1.   1.   1.]]
mask :[[  0. -inf -inf -inf]
 [  0.   0. -inf -inf]
 [  0.   0.   0. -inf]
 [  0.   0.   0.   0.]]


In [6]:
#--apply softmax on with and without mask
def softmax(x):
    return np.exp(x)/np.sum(np.exp(x),axis=-1)

In [7]:
print(f'scaled :{scaled}')
print(f'scaled + mask  :{scaled+mask}')


scaled :[[-1.94553837 -0.09040649 -2.48013571  1.43851704]
 [ 0.23939813  0.16373961 -0.41752202  0.56765996]
 [ 0.46849232  1.06915709 -1.02662752  0.82210341]
 [ 0.51040561 -1.30518591  0.94916364  1.14090451]]
scaled + mask  :[[-1.94553837        -inf        -inf        -inf]
 [ 0.23939813  0.16373961        -inf        -inf]
 [ 0.46849232  1.06915709 -1.02662752        -inf]
 [ 0.51040561 -1.30518591  0.94916364  1.14090451]]


In [8]:
softmax_without_mask=softmax(scaled)
softmax_with_mask=softmax(scaled+mask)
print(f'softmax_without_mask :{softmax_without_mask}')
print(f'softmax_with_mask :{softmax_with_mask}')

softmax_without_mask :[[0.02668904 0.18754295 0.01172058 0.55089046]
 [0.23726777 0.24181041 0.09220002 0.23059869]
 [0.29835484 0.59798847 0.05014179 0.29741318]
 [0.31112564 0.05565815 0.36163883 0.4090852 ]]
softmax_with_mask :[[ 1.          0.          0.          0.        ]
 [ 8.89008403  0.48109439  0.          0.        ]
 [11.17892924  1.18972916  0.07357437  0.        ]
 [11.65743272  0.11073478  0.53064215  0.4090852 ]]


In [9]:
#--create a single function
def scaled_dot_product(q,k,v,mask=None):
    d_k=q.shape[1]
    print(f'd_k :{d_k}')
    
    dot_product=np.matmul(q,k.T)
    scaled_dot_product=dot_product/math.sqrt(d_k)
    print(f'dot_product var :{dot_product.var()}')
    print(f'scaled_dot_product var :{scaled_dot_product.var()}')

    #---apply masking
    if mask is not None:
        scaled_dot_product=scaled_dot_product+mask

    #--apply softmax
    new_attention=softmax(scaled_dot_product)

    #--get new values
    new_values=np.matmul(new_attention,v)

    return new_attention, new_values

In [10]:
mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

In [11]:
# new_attention, new_values=scaled_dot_product(q,k,v,mask=None)
new_attention, new_values=scaled_dot_product(q,k,v,mask=mask)

print(f'new_attention :{new_attention} with shape :{new_attention.shape}') #--> (seq_len, seq_len)
print(f'new_values :{new_values} with shape :{new_values.shape}') #--> (seq_len, v)


d_k :8
dot_product var :9.926351575145791
scaled_dot_product var :1.2407939468932236
new_attention :[[ 1.          0.          0.          0.        ]
 [ 8.89008403  0.48109439  0.          0.        ]
 [11.17892924  1.18972916  0.07357437  0.        ]
 [11.65743272  0.11073478  0.53064215  0.4090852 ]] with shape :(4, 4)
new_values :[[ -0.63364004  -1.20415667   0.95140385  -1.30287788   0.35851064
    0.86457743  -0.21113271   0.11354354]
 [ -5.20441469  -9.70758297   9.13446328 -10.99418014   3.33614879
    8.42652864  -2.68755318   1.50354095]
 [ -5.97721014 -11.01554263  12.3375955  -13.12120184   4.47908486
   11.40837388  -4.42914829   2.64093788]
 [ -6.89876969 -14.41306818  11.77943072 -15.11050648   5.37424631
   10.01081229  -3.36968758   2.00496132]] with shape :(4, 8)
